<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/KVCompass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KVCompass Benchmark Team Notebook

This notebook is organized around the **core benchmark matrix**: RULER task category x method x budget x context length. Each teammate should run only their labeled section, then Aarnav can use the final aggregation cells to combine the benchmark summaries for the presentation.


## Shared Experiment Plan

**Benchmark**
- `RULER`
- `fraction: 0.02`
- context lengths: `4096`, `8192`

**Task categories**
- Needle In A Haystack: `niah`
- Question Answering: `qa`
- Multi-Hop Tracing: `vt`
- Aggregation: `cwe`, `fwe`

**Methods in the core matrix**
- `no_compression @ 1.0`
- `snapkv @ 0.5`
- `expected_attention @ 0.5`
- `knorm @ 0.5`
- `streaming_llm @ 0.5`

**Team split**
- Tony: `niah` compressed-method runs at `4096` and `8192`
- Will: `qa` compressed-method runs at `4096` and `8192`
- Grady: `vt` compressed-method runs at `4096` and `8192`
- Jamez: all `no_compression` baselines across all categories and both context lengths
- Aarnav: `aggregation` compressed-method runs at `4096` and `8192`, plus final aggregation

**Core matrix size**
- `40` total benchmark runs
- `32` compressed-method runs across the four category owners
- `8` baseline runs for Jamez


In [ ]:
# Shared setup: clone/refresh the repo, switch to the demo branch, mount Drive, and install dependencies.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BRANCH_NAME = 'codex/colab-sweep-workflow'
repo_dir = Path('/content/KVCompass')
if not repo_dir.exists():
    !git clone https://github.com/AarnavSawant/KVCompass.git /content/KVCompass

%cd /content/KVCompass
!git fetch origin
!git checkout "$BRANCH_NAME"
!git pull origin "$BRANCH_NAME"
!nvidia-smi
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .


In [ ]:
# Shared auth: set HF_TOKEN from Colab secrets and verify the login.
from google.colab import userdata
from huggingface_hub import whoami
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print(whoami())


In [ ]:
# Shared values used by all run cells.
from pathlib import Path

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
FRACTION = 0.02
TORCH_DTYPE = 'bfloat16'
SHARED_RESULTS_DIR = Path('/content/drive/MyDrive/KVCompass')
SHARED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Shared results dir:', SHARED_RESULTS_DIR)


## Tony: Needle In A Haystack

Run the next two cells only. This assignment covers the benchmark-backed `niah` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 1.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_1
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: needle_in_a_haystack_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
    - name: needle_in_a_haystack_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_1.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_1.yaml').read_text())


In [ ]:
# Run Assignment 1.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_1.yaml


## Will: Question Answering

Run the next two cells only. This assignment covers the benchmark-backed `qa` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 2.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_2
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: question_answering_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
    - name: question_answering_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_2.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_2.yaml').read_text())


In [ ]:
# Run Assignment 2.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_2.yaml


## Grady: Multi-Hop Tracing

Run the next two cells only. This assignment covers the benchmark-backed `vt` slice for the compressed methods at `4096` and `8192`.


In [ ]:
# Write the sweep config for Assignment 3.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_3
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: multi_hop_tracing_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
    - name: multi_hop_tracing_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_3.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_3.yaml').read_text())


In [ ]:
# Run Assignment 3.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_3.yaml


## Jamez: No Compression Baselines

Run the next two cells only. This assignment covers the `no_compression` baselines across all four categories and both context lengths.


In [ ]:
# Write the sweep config for Assignment 4.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_4
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: niah_4k_baseline
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: niah_8k_baseline
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["niah"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: qa_4k_baseline
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: qa_8k_baseline
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["qa"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: vt_4k_baseline
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: vt_8k_baseline
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["vt"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: aggregation_4k_baseline
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
    - name: aggregation_8k_baseline
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods: [no_compression]
      budgets:
        no_compression: [1.0]
"""
Path('configs/benchmark_sweeps.assignment_4.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_4.yaml').read_text())


In [ ]:
# Run Assignment 4.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_4.yaml


## Aarnav: Aggregation And Final Aggregation

Run the next two cells for the benchmark-backed `cwe, fwe` aggregation slice for the compressed methods at `4096` and `8192`. Then use the final aggregation cells after everyone has shared their outputs.


In [ ]:
# Write the sweep config for Assignment 5.
from pathlib import Path

config_text = f"""
sweep:
  name: assignment_5
  model: {MODEL_NAME}
  device: auto
  torch_dtype: {TORCH_DTYPE}
  methods_config_path: configs/methods.yaml
  output_dir: {SHARED_RESULTS_DIR.as_posix()}/benchmark_eval
  seed: 42
  verbose: false
  scenarios:
    - name: aggregation_4k
      dataset: ruler
      data_dir: "4096"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
    - name: aggregation_8k
      dataset: ruler
      data_dir: "8192"
      task_prefixes: ["cwe", "fwe"]
      fraction: {FRACTION}
      methods:
        - snapkv
        - expected_attention
        - knorm
        - streaming_llm
      budgets:
        default: [0.5]
"""
Path('configs/benchmark_sweeps.assignment_5.yaml').write_text(config_text, encoding='utf-8')
print(Path('configs/benchmark_sweeps.assignment_5.yaml').read_text())


In [ ]:
# Run Assignment 5.
!python scripts/run_kvpress_benchmark_sweep.py --config configs/benchmark_sweeps.assignment_5.yaml


## Final Aggregation For The Presentation

After everyone finishes, make sure all of the `assignment_*__summary.csv` files are present under `results/benchmark_eval`, then run the next cells.


In [ ]:
# Load and combine the benchmark assignment summaries.
import json
from pathlib import Path
import pandas as pd

summary_root = SHARED_RESULTS_DIR / 'benchmark_eval'
summary_paths = [summary_root / f'assignment_{i}__summary.csv' for i in range(1, 6) if (summary_root / f'assignment_{i}__summary.csv').exists()]
combined_summary = pd.concat([pd.read_csv(path) for path in summary_paths], ignore_index=True) if summary_paths else pd.DataFrame()
print('Summary files:', [str(path) for path in summary_paths])
display(combined_summary)

if not combined_summary.empty:
    rows = []
    for _, row in combined_summary.iterrows():
        metrics = json.loads(Path(row['metrics_path']).read_text())
        metric_values = []
        for task_metrics in metrics.values():
            if isinstance(task_metrics, dict):
                metric_values.extend(v for v in task_metrics.values() if isinstance(v, (int, float)))
        rows.append({
            'scenario_name': row['scenario_name'],
            'dataset': row['dataset'],
            'data_dir': row['data_dir'],
            'task_prefixes': row['task_prefixes'],
            'method': row['method'],
            'budget': row['budget'],
            'avg_quality': round(sum(metric_values) / len(metric_values), 2) if metric_values else None,
            'avg_latency_seconds': row.get('avg_latency_seconds'),
            'avg_throughput_tokens_per_second': row.get('avg_throughput_tokens_per_second'),
            'peak_gpu_memory_mb': row.get('peak_gpu_memory_mb'),
        })
    leaderboard = pd.DataFrame(rows).sort_values(['scenario_name', 'avg_quality', 'avg_latency_seconds'], ascending=[True, False, True])
    display(leaderboard)
else:
    print('No assignment summaries found yet.')


In [ ]:
# Build a presentation-friendly matrix view.
import pandas as pd

if 'leaderboard' in globals() and not leaderboard.empty:
    matrix_view = leaderboard.pivot_table(
        index=['scenario_name', 'data_dir'],
        columns='method',
        values='avg_quality',
        aggfunc='first',
    )
    display(matrix_view)
else:
    print('Leaderboard not ready yet.')


In [ ]:
# Presentation plots.
import matplotlib.pyplot as plt

if 'leaderboard' in globals() and not leaderboard.empty:
    plot_df = leaderboard.copy()
    plot_df['label'] = plot_df['scenario_name'] + ' | ' + plot_df['method'] + ' @ ' + plot_df['budget'].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    axes[0].bar(plot_df['label'], plot_df['avg_quality'], color='#2a6f97')
    axes[0].set_title('Average Benchmark Quality')
    axes[0].tick_params(axis='x', rotation=90)
    axes[1].bar(plot_df['label'], plot_df['avg_latency_seconds'], color='#c1121f')
    axes[1].set_title('Average Latency (s)')
    axes[1].tick_params(axis='x', rotation=90)
    axes[2].bar(plot_df['label'], plot_df['peak_gpu_memory_mb'], color='#6a994e')
    axes[2].set_title('Peak GPU Memory (MB)')
    axes[2].tick_params(axis='x', rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print('Leaderboard not ready yet.')


In [ ]:
# Simple recommendation table.
import pandas as pd

if 'leaderboard' in globals() and not leaderboard.empty:
    best_quality = leaderboard.sort_values(['avg_quality', 'avg_latency_seconds'], ascending=[False, True]).iloc[0]
    best_latency = leaderboard.sort_values('avg_latency_seconds', ascending=True).iloc[0]
    best_memory = leaderboard.dropna(subset=['peak_gpu_memory_mb']).sort_values('peak_gpu_memory_mb', ascending=True).iloc[0]
    recommendations = pd.DataFrame([
        {'category': 'best_for_quality', 'scenario_name': best_quality['scenario_name'], 'method': best_quality['method'], 'budget': best_quality['budget']},
        {'category': 'best_for_latency', 'scenario_name': best_latency['scenario_name'], 'method': best_latency['method'], 'budget': best_latency['budget']},
        {'category': 'best_for_memory', 'scenario_name': best_memory['scenario_name'], 'method': best_memory['method'], 'budget': best_memory['budget']},
    ])
    display(recommendations)
else:
    print('Leaderboard not ready yet.')


In [ ]:
# Optional smoke test: one tiny benchmark-backed slice to verify the environment.
!python scripts/run_kvpress_benchmark_eval.py   --dataset ruler   --data-dir 4096   --model Qwen/Qwen2.5-1.5B-Instruct   --method no_compression   --budget 1.0   --fraction 0.002   --torch-dtype bfloat16   --output-dir "$SHARED_RESULTS_DIR/benchmark_eval"
